## PageRank with Snowpark on Snowflake


In [ ]:
from typing import Tuple

from snowflake.snowpark import functions as F
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.dataframe import DataFrame

session = get_active_session()

In [ ]:
DEFAULT_DAMPING_FACTOR = 0.85
DEFAULT_ITERATIONS = 10
TOP_N = 20

INPUT_TABLE = "BIGDATA_DB.STAGING.TRANSFERS_INDEXED"
OUTPUT_TABLE = "BIGDATA_DB.STAGING.PAGERANK_FINAL_RESULTS"
TRUSTED_TABLE = "BIGDATA_DB.STAGING.TRUSTED_ADDRESS"
MAPPING_TABLE = "BIGDATA_DB.STAGING.MAP_USER"

In [ ]:
df_transfers = session.table(INPUT_TABLE)
df_transfers.limit(5).show()

In [ ]:
def build_graph(transfers: DataFrame) -> Tuple[DataFrame, DataFrame, DataFrame]:
    edges = (
        transfers.select(
            F.col("from_user_id").alias("src"),
            F.col("to_user_id").alias("dst"),
        )
        .filter(
            F.col("src").is_not_null()
            & F.col("dst").is_not_null()
            & (F.col("src") != F.col("dst"))
        )
        .distinct()
    )

    vertices = (
        edges.select(F.col("src").alias("id"))
        .union_all(edges.select(F.col("dst").alias("id")))
        .distinct()
    )

    out_degrees = edges.group_by("src").agg(F.count("*").alias("out_degree"))
    return edges, vertices, out_degrees

In [ ]:
def save_results(ranks: DataFrame, output_table: str) -> None:
    (
        ranks.select(
            F.col("id").cast("int").alias("id"),
            F.col("pagerank").cast("double").alias("pagerank"),
        )
        .write.mode("overwrite")
        .save_as_table(output_table)
    )

In [ ]:
def compute_trustrank(
    transfers: DataFrame,
    iterations: int = DEFAULT_ITERATIONS,
    damping_factor: float = DEFAULT_DAMPING_FACTOR,
) -> DataFrame:
    if not 0.0 < damping_factor < 1.0:
        raise ValueError("damping_factor must be between 0 and 1.")
    if iterations < 1:
        raise ValueError("iterations must be >= 1.")

    edges, vertices_raw, out_degrees = build_graph(transfers)
    num_vertices = vertices_raw.count()
    if num_vertices == 0:
        raise ValueError("Input graph has no valid edges or vertices.")

    edge_count = edges.count()
    
    # 1. Load Trusted Nodes và Mapping từ String Address sang Integer ID
    df_trusted_raw = session.table(TRUSTED_TABLE)
    df_mapping = session.table(MAPPING_TABLE)
    
    df_trusted = (
        df_trusted_raw.join(
            df_mapping, 
            df_trusted_raw["address"] == df_mapping["user_address"]
        )
        .select(df_mapping["user_id"].alias("trusted_id"))
        .distinct()
    )
    
    num_trusted = df_trusted.count()
    if num_trusted == 0:
        raise ValueError("No trusted addresses found after mapping. Check your MAPPING_TABLE.")
        
    print(f"Graph ready: {num_vertices:,} vertices, {edge_count:,} edges. Trusted Nodes (Mapped): {num_trusted:,}")

    # Gắn cờ is_trusted cho các đỉnh
    vertices = (
        vertices_raw.join(df_trusted, vertices_raw["id"] == df_trusted["trusted_id"], join_type="left")
        .select(
            vertices_raw["id"].alias("id"),
            F.when(df_trusted["trusted_id"].is_not_null(), F.lit(1)).otherwise(F.lit(0)).alias("is_trusted")
        )
    )

    # 2. Khởi tạo điểm: Chỉ chia điểm cho Trusted Nodes
    ranks = vertices.select(
        F.col("id"),
        F.col("is_trusted"),
        F.when(F.col("is_trusted") == 1, F.lit(1.0 / num_trusted).cast("double"))
        .otherwise(F.lit(0.0)).alias("pagerank")
    )

    out_degrees_for_edges = out_degrees.select(
        F.col("src").alias("degree_src"),
        F.col("out_degree"),
    )

    edges_with_out_degree = edges.join(
        out_degrees_for_edges,
        edges["src"] == out_degrees_for_edges["degree_src"],
    ).select(
        edges["src"].alias("src"),
        edges["dst"].alias("dst"),
        out_degrees_for_edges["out_degree"].alias("out_degree"),
    )

    # 3. Vòng lặp Power Iteration
    base_rank_trusted = (1.0 - damping_factor) / num_trusted

    for iteration in range(1, iterations + 1):
        out_degrees_for_ranks = out_degrees.select(
            F.col("src").alias("degree_src"),
            F.col("out_degree"),
        )

        ranks_with_out_degree = ranks.join(
            out_degrees_for_ranks,
            ranks["id"] == out_degrees_for_ranks["degree_src"],
            join_type="left",
        ).select(
            ranks["id"].alias("id"),
            ranks["pagerank"].alias("pagerank"),
            out_degrees_for_ranks["out_degree"].alias("out_degree"),
        )

        # Tính toán Dangling Mass (Điểm kẹt)
        dangling_row = (
            ranks_with_out_degree.filter(F.col("out_degree").is_null())
            .agg(F.coalesce(F.sum("pagerank"), F.lit(0.0)).alias("dangling_mass"))
            .collect()[0]
        )
        dangling_mass = float(dangling_row["DANGLING_MASS"] or 0.0)
        
        # Điểm kẹt chỉ được chia lại cho Trusted Nodes
        dangling_share_trusted = dangling_mass / num_trusted

        # Tính tổng điểm nhận được từ các cạnh trỏ tới
        contributions = (
            edges_with_out_degree.join(ranks, edges_with_out_degree["src"] == ranks["id"])
            .select(
                edges_with_out_degree["dst"].alias("contrib_id"),
                (ranks["pagerank"] / edges_with_out_degree["out_degree"]).alias("contrib"),
            )
            .group_by("contrib_id")
            .agg(F.sum("contrib").alias("sum_contrib"))
        )

        # Cập nhật điểm Ranks mới
        trust_jump = F.lit(base_rank_trusted) + (F.lit(damping_factor) * F.lit(dangling_share_trusted))
        
        ranks = (
            vertices.join(
                contributions,
                vertices["id"] == contributions["contrib_id"],
                join_type="left",
            )
            .select(
                vertices["id"].alias("id"),
                vertices["is_trusted"].alias("is_trusted"),
                (
                    F.lit(damping_factor) * F.coalesce(contributions["sum_contrib"], F.lit(0.0))
                    + F.when(vertices["is_trusted"] == 1, trust_jump).otherwise(F.lit(0.0))
                ).alias("pagerank"),
            )
        )

        print(
            f"Iteration {iteration}/{iterations} finished "
            f"(dangling_mass={dangling_mass:.8f})"
        )

    return ranks

In [ ]:
ranks = compute_trustrank(
    transfers=df_transfers,
    iterations=DEFAULT_ITERATIONS,
    damping_factor=DEFAULT_DAMPING_FACTOR,
).order_by(F.desc("pagerank"))

save_results(ranks, OUTPUT_TABLE)
print(f"Saved TrustRank results to: {OUTPUT_TABLE}")